# 単一Prefixの詳細分析

selected native Prefixを1つ指定し、Raw、Broad removed、Broad remainingの観測事実を比較します。PCAPの再解析やpipelineの実行は行わず、成功済みrun manifestに登録されたcanonical artifactだけを読みます。canonical `src_ip` / `dst_ip` とmembershipの `src_match` / `dst_match` は観測方向の事実であり、接続のinitiator/responderを表しません。TCPの接続先候補は、plain SYNを観測したflowの `initial_syn_receiver_ip` / `initial_syn_receiver_port` からのみ確認します。

## 解析設定

In [ ]:
DATASET_ID = "202604081400"
RUN_NAME = "n3_m2"
TARGET_PREFIX = "163.29.158.43/32"

TOP_N_PORTS = 10
TOP_N_TARGETS = 10


## Artifact読み込みとprovenance

In [ ]:
from pathlib import Path
import ipaddress
import json
import os

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

from mawi_global_analysis.comparison import build_comparison_flow_inclusion
from mawi_global_analysis.io import load_run

japanese_font_candidates = ('Hiragino Sans', 'Noto Sans CJK JP', 'Noto Serif CJK JP', 'IPAexGothic', 'YuGothic', 'Meiryo')
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
japanese_font = next((font for font in japanese_font_candidates if font in available_font_names), None)
if japanese_font is None:
    raise RuntimeError('日本語グリフを含むMatplotlibフォントが見つかりません')
plt.rcParams['font.family'] = japanese_font
plt.rcParams['axes.unicode_minus'] = False

def resolve_analysis_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'mawi_global_analysis').is_dir():
            return candidate
    return Path.cwd()

root = Path(os.environ.get('MAWI_ANALYSIS_ROOT', resolve_analysis_root())).resolve()
run = load_run(DATASET_ID, RUN_NAME, root=root)
saved_config = yaml.safe_load(run.manifest.get('config', {}).get('text', ''))
if not isinstance(saved_config, dict):
    raise ValueError('run manifest内の保存済みconfigを読み取れません')
# scan.broad.enabled must be true for this Raw/Broad deep dive.
if saved_config.get('scan', {}).get('broad', {}).get('enabled') is not True:
    raise ValueError('指定runはscan.broad.enabled == Trueではありません。Broad有効の成功済みrunを指定してください')

artifact_paths = run.manifest.get('artifacts', {})
provenance = pd.DataFrame([{
    'データセット': DATASET_ID,
    '実行名': RUN_NAME,
    '対象native Prefix': TARGET_PREFIX,
    'manifest status': run.manifest.get('status'),
    'config hash': run.manifest.get('config', {}).get('hash'),
    'input SHA-256': run.manifest.get('input', {}).get('sha256'),
    'Git commit': run.manifest.get('git_commit'),
    'flows artifact': artifact_paths.get('flows', {}).get('path'),
    'flow_labels artifact': artifact_paths.get('flow_labels', {}).get('path'),
    'membership artifact': artifact_paths.get('flow_prefix_membership', {}).get('path'),
}])
display(provenance)


## Prefix validation

In [ ]:
selected_native_prefixes = (
    run.prefixes.loc[run.prefixes['selected_for_analysis'] == True, 'prefix']
    .drop_duplicates()
    .sort_values()
    .tolist()
)
if TARGET_PREFIX not in selected_native_prefixes:
    available = '\n'.join(f'  - {prefix}' for prefix in selected_native_prefixes)
    raise ValueError(
        f'指定したTARGET_PREFIXはselected native Prefixに存在しません: {TARGET_PREFIX}\n'
        f'利用可能なselected native Prefix:\n{available}'
    )
display(pd.DataFrame({'selected native Prefix': selected_native_prefixes}))


## 対象flow DataFrameの構築

In [ ]:
required_flow_columns = {
    'flow_id', 'ip_version', 'protocol', 'packet_count', 'byte_count', 'frame_byte_count',
    'ip_byte_count', 'transport_payload_byte_count', 'duration', 'observed_tcp_pattern',
    'initial_syn_sender_ip', 'initial_syn_sender_port',
    'initial_syn_receiver_ip', 'initial_syn_receiver_port',
}
missing_flow_columns = sorted(required_flow_columns - set(run.flows.columns))
if missing_flow_columns:
    raise ValueError(f'flowsに必要な列がありません: {missing_flow_columns}')
if run.flows['flow_id'].duplicated().any():
    raise ValueError('flows.flow_idが一意ではありません')
if run.labels['flow_id'].duplicated().any():
    raise ValueError('labels.flow_idが一意ではありません')
if set(run.flows['flow_id']) != set(run.labels['flow_id']):
    raise ValueError('flowsとlabelsのflow_id集合が一致しません')

# membership全体を結合せず、単一native Prefixを先に絞ることで行増殖を防ぐ。
target_membership = run.membership.loc[
    run.membership['analysis_scope'].eq('native')
    & run.membership['analysis_prefix'].eq(TARGET_PREFIX)
].copy()
if target_membership.empty:
    raise ValueError(f'TARGET_PREFIXのnative membershipがありません: {TARGET_PREFIX}')
if target_membership['flow_id'].duplicated().any():
    raise ValueError('target membershipのflow_idが一意ではありません')
if not set(target_membership['flow_id']).issubset(set(run.flows['flow_id'])):
    raise ValueError('target membershipにflows外のflow_idが含まれます')

inclusion = build_comparison_flow_inclusion(run.flows, run.labels)
flow_facts = run.flows.merge(
    run.labels.loc[:, ['flow_id', 'broad_removed']], on='flow_id', how='inner', validate='one_to_one'
).merge(
    inclusion, on='flow_id', how='inner', validate='one_to_one'
)
target_flows = target_membership.merge(
    flow_facts, on='flow_id', how='inner', validate='one_to_one'
).copy()
if len(target_flows) != len(target_membership):
    raise ValueError('target flow joinで意図しない行数変化が発生しました')
target_flows['broad_removed'] = target_flows['broad_removed'].astype(bool)
if not target_flows['raw_included'].all() or not (target_flows['broad_included'] == ~target_flows['broad_removed']).all():
    raise ValueError('Raw/Broad inclusionとbroad_removedの対応が不正です')
target_flows['group'] = np.where(target_flows['broad_removed'], 'Broad removed', 'Broad remaining')
raw_target_flows = target_flows
broad_removed_flows = target_flows.loc[target_flows['broad_removed']].copy()
broad_remaining_flows = target_flows.loc[target_flows['broad_included']].copy()
if len(broad_removed_flows) + len(broad_remaining_flows) != len(raw_target_flows):
    raise AssertionError('Broad removed + Broad remaining == Raw')
if not (pd.to_numeric(target_flows['byte_count'], errors='raise') == pd.to_numeric(target_flows['frame_byte_count'], errors='raise')).all():
    raise ValueError('byte_countとframe_byte_countが同値ではありません')

groups = {
    'Raw': raw_target_flows,
    'Broad removed': broad_removed_flows,
    'Broad remaining': broad_remaining_flows,
}
display(pd.DataFrame([{
    'TARGET_PREFIX': TARGET_PREFIX,
    'native membership行数': len(target_membership),
    'Raw flow数': len(raw_target_flows),
    'Broad removed + Broad remaining == Raw': True,
}]))


## Basic Raw / Broad statistics

In [ ]:
def ratio_percent(values):
    return float(values.mean() * 100) if len(values) else np.nan

def numeric_median(frame, column):
    return pd.to_numeric(frame[column], errors='coerce').median() if len(frame) else np.nan

raw_median_packet_count = numeric_median(raw_target_flows, 'packet_count')
remaining_median_packet_count = numeric_median(broad_remaining_flows, 'packet_count')
basic_statistics = pd.DataFrame([{
    'Prefix': TARGET_PREFIX,
    'Raw flow count': len(raw_target_flows),
    'Broad removed flow count': len(broad_removed_flows),
    'Broad remaining flow count': len(broad_remaining_flows),
    'Broad removed ratio [%]': ratio_percent(raw_target_flows['broad_removed']),
    'Raw median packet_count': raw_median_packet_count,
    'Broad remaining median packet_count': remaining_median_packet_count,
    'packet_count median change rate [%]': ((remaining_median_packet_count - raw_median_packet_count) / raw_median_packet_count * 100) if pd.notna(raw_median_packet_count) and raw_median_packet_count != 0 else np.nan,
    'Removed median packet_count': numeric_median(broad_removed_flows, 'packet_count'),
    'Remaining median packet_count': remaining_median_packet_count,
    'Raw one-packet ratio [%]': ratio_percent(raw_target_flows['packet_count'].eq(1)),
    'Removed one-packet ratio [%]': ratio_percent(broad_removed_flows['packet_count'].eq(1)),
    'Remaining one-packet ratio [%]': ratio_percent(broad_remaining_flows['packet_count'].eq(1)),
    'Raw tiny-flow (<=3) ratio [%]': ratio_percent(raw_target_flows['packet_count'].le(3)),
    'Removed tiny-flow (<=3) ratio [%]': ratio_percent(broad_removed_flows['packet_count'].le(3)),
    'Remaining tiny-flow (<=3) ratio [%]': ratio_percent(broad_remaining_flows['packet_count'].le(3)),
}])
display(basic_statistics)


## packet_count分析

In [ ]:
def packet_distribution_summary(name, frame):
    packets = pd.to_numeric(frame['packet_count'], errors='coerce').dropna()
    return {
        'group': name, 'count': len(packets), 'Q25': packets.quantile(.25) if len(packets) else np.nan,
        'median': packets.median() if len(packets) else np.nan, 'Q75': packets.quantile(.75) if len(packets) else np.nan,
        'Q90': packets.quantile(.90) if len(packets) else np.nan, 'Q99': packets.quantile(.99) if len(packets) else np.nan,
        'max': packets.max() if len(packets) else np.nan,
        'packet_count == 1 ratio [%]': ratio_percent(packets.eq(1)),
        'packet_count <= 3 ratio [%]': ratio_percent(packets.le(3)),
    }

packet_distribution = pd.DataFrame([packet_distribution_summary(name, frame) for name, frame in groups.items()])
display(packet_distribution)

def ecdf(values):
    ordered = np.sort(np.asarray(values, dtype=float))
    return ordered, np.arange(1, len(ordered) + 1) / len(ordered)

figure, axis = plt.subplots(figsize=(8, 4.8))
for name, frame in groups.items():
    packets = pd.to_numeric(frame['packet_count'], errors='coerce').dropna()
    if len(packets):
        x, y = ecdf(packets)
        axis.step(x, y, where='post', label=name)
axis.set(title=f'{TARGET_PREFIX}: packet_countのECDF', xlabel='packet_count', ylabel='累積確率', xscale='log', ylim=(0, 1))
axis.legend()
figure.tight_layout()


## TCP pattern分析

In [ ]:
PATTERN_ORDER = ['syn_only_observed', 'syn_to_rst', 'syn_synack_rst', 'その他', 'none']

def is_tcp(frame):
    return frame['protocol'].astype(str).isin({'6', '6.0', 'tcp'})

def pattern_bucket(value):
    return value if value in {'syn_only_observed', 'syn_to_rst', 'syn_synack_rst', 'none'} else 'その他'

pattern_rows = []
for name, frame in groups.items():
    tcp = frame.loc[is_tcp(frame)].copy()
    bucketed = tcp['observed_tcp_pattern'].fillna('none').map(pattern_bucket)
    for pattern in PATTERN_ORDER:
        count = int(bucketed.eq(pattern).sum())
        pattern_rows.append({'group': name, 'TCP flow count': len(tcp), 'observed_tcp_pattern': pattern, 'flow count': count, 'ratio within TCP flows [%]': count / len(tcp) * 100 if len(tcp) else np.nan})
tcp_pattern_comparison = pd.DataFrame(pattern_rows)
display(tcp_pattern_comparison)


## 接続先 / 推定サービス候補

In [ ]:
def tcp_with_initial_syn(frame):
    return frame.loc[
        is_tcp(frame)
        & frame['initial_syn_receiver_ip'].notna()
        & frame['initial_syn_receiver_port'].notna()
    ].copy()

def service_candidate(port):
    port = int(port)
    known = {22: 'SSH関連サービス候補', 80: 'HTTP関連サービス候補', 443: 'HTTPS/TLS関連サービス候補', 53: 'DNS関連サービス候補', 25: 'SMTP関連サービス候補'}
    if port in known:
        return f'TCP/{port}: {known[port]}'
    if port >= 49152:
        return f'TCP/{port}: dynamic/high port（port番号だけでは用途不明）'
    return f'TCP/{port}: port番号だけでは用途不明'

def receiver_port_summary(name, frame):
    syn_tcp = tcp_with_initial_syn(frame)
    if syn_tcp.empty:
        return pd.DataFrame(columns=['group', 'receiver port', 'flow count', 'group ratio [%]', 'median packet_count', 'median duration', 'unique receiver IP count', 'Service candidate'])
    result = syn_tcp.groupby('initial_syn_receiver_port', as_index=False).agg(
        **{'flow count': ('flow_id', 'size'), 'median packet_count': ('packet_count', 'median'), 'median duration': ('duration', 'median'), 'unique receiver IP count': ('initial_syn_receiver_ip', 'nunique')}
    ).sort_values(['flow count', 'initial_syn_receiver_port'], ascending=[False, True]).head(TOP_N_PORTS).rename(columns={'initial_syn_receiver_port': 'receiver port'})
    result.insert(0, 'group', name)
    result['group ratio [%]'] = result['flow count'] / len(syn_tcp) * 100
    result['Service candidate'] = result['receiver port'].map(service_candidate)
    return result

receiver_port_comparison = pd.concat([
    receiver_port_summary('Broad removed', broad_removed_flows),
    receiver_port_summary('Broad remaining', broad_remaining_flows),
], ignore_index=True)
display(receiver_port_comparison)


## Receiver IP / target diversity

In [ ]:
diversity_rows = []
for name, frame in groups.items():
    syn_tcp = tcp_with_initial_syn(frame)
    diversity_rows.append({
        'group': name, 'TCP flows with observed plain SYN': len(syn_tcp),
        'unique initial_syn_receiver_ip': syn_tcp['initial_syn_receiver_ip'].nunique(),
        'unique initial_syn_receiver_port': syn_tcp['initial_syn_receiver_port'].nunique(),
        'unique (receiver IP, receiver port)': syn_tcp.loc[:, ['initial_syn_receiver_ip', 'initial_syn_receiver_port']].drop_duplicates().shape[0],
    })
target_diversity = pd.DataFrame(diversity_rows)
display(target_diversity)

removed_syn_tcp = tcp_with_initial_syn(broad_removed_flows)
top_removed_receiver_ips = removed_syn_tcp.groupby('initial_syn_receiver_ip', as_index=False).agg(
    **{'flow count': ('flow_id', 'size'), 'unique receiver ports': ('initial_syn_receiver_port', 'nunique'), 'median packet_count': ('packet_count', 'median')}
).sort_values(['flow count', 'initial_syn_receiver_ip'], ascending=[False, True]).head(TOP_N_TARGETS)
top_removed_receiver_pairs = removed_syn_tcp.groupby(['initial_syn_receiver_ip', 'initial_syn_receiver_port'], as_index=False).agg(
    **{'flow count': ('flow_id', 'size'), 'median packet_count': ('packet_count', 'median'), 'median duration': ('duration', 'median')}
).sort_values(['flow count', 'initial_syn_receiver_ip', 'initial_syn_receiver_port'], ascending=[False, True, True]).head(TOP_N_TARGETS)
display(top_removed_receiver_ips)
display(top_removed_receiver_pairs)


## byte / duration / payload分析

In [ ]:
def byte_duration_payload_summary(name, frame):
    return {
        'group': name, 'flow count': len(frame),
        'total frame_byte_count': pd.to_numeric(frame['frame_byte_count'], errors='coerce').sum(),
        'median frame_byte_count': numeric_median(frame, 'frame_byte_count'),
        'total ip_byte_count': pd.to_numeric(frame['ip_byte_count'], errors='coerce').sum(),
        'median ip_byte_count': numeric_median(frame, 'ip_byte_count'),
        'total transport_payload_byte_count': pd.to_numeric(frame['transport_payload_byte_count'], errors='coerce').sum(),
        'median transport_payload_byte_count': numeric_median(frame, 'transport_payload_byte_count'),
        'transport_payload_byte_count == 0 ratio [%]': ratio_percent(frame['transport_payload_byte_count'].eq(0)),
        'median duration': numeric_median(frame, 'duration'),
        'Q90 duration': pd.to_numeric(frame['duration'], errors='coerce').quantile(.90) if len(frame) else np.nan,
    }

byte_duration_payload = pd.DataFrame([byte_duration_payload_summary(name, frame) for name, frame in groups.items()])
display(byte_duration_payload)
display(pd.DataFrame([{'byte_count == frame_byte_count': True, '注記': 'byte_countはlegacy互換のEthernet frame byte totalのため、重複統計は表示しない'}]))


## Prefix側のinitial SYN role

In [ ]:
target_network = ipaddress.ip_network(TARGET_PREFIX)

def endpoint_in_target(value):
    return pd.notna(value) and ipaddress.ip_address(str(value)) in target_network

def syn_role(frame):
    sender_in_target = frame['initial_syn_sender_ip'].map(endpoint_in_target)
    receiver_in_target = frame['initial_syn_receiver_ip'].map(endpoint_in_target)
    return np.select(
        [sender_in_target & ~receiver_in_target, receiver_in_target & ~sender_in_target, sender_in_target & receiver_in_target],
        ['initial SYN sender', 'initial SYN receiver', 'both'],
        default='unknown',
    )

role_rows = []
role_order = ['initial SYN sender', 'initial SYN receiver', 'both', 'unknown']
for name, frame in groups.items():
    syn_tcp = tcp_with_initial_syn(frame)
    roles = pd.Series(syn_role(syn_tcp), index=syn_tcp.index) if len(syn_tcp) else pd.Series(dtype='object')
    for role in role_order:
        count = int(roles.eq(role).sum())
        role_rows.append({'group': name, '母集団: initial SYN観測TCP flow数': len(syn_tcp), 'Prefix-side SYN role': role, 'flow count': count, 'ratio within observed-SYN TCP [%]': count / len(syn_tcp) * 100 if len(syn_tcp) else np.nan})
syn_role_comparison = pd.DataFrame(role_rows)
display(syn_role_comparison)


## Removed-flow audit

In [ ]:
removed_flow_audit_columns = [
    'flow_id', 'protocol', 'observed_tcp_pattern',
    'initial_syn_sender_ip', 'initial_syn_sender_port',
    'initial_syn_receiver_ip', 'initial_syn_receiver_port',
    'packet_count', 'frame_byte_count', 'transport_payload_byte_count', 'duration',
]
removed_flow_audit = broad_removed_flows.loc[:, removed_flow_audit_columns].sort_values('flow_id', kind='stable').reset_index(drop=True)
display(pd.DataFrame([{'Broad removed flow数': len(removed_flow_audit), 'Notebook上の表示行数': min(20, len(removed_flow_audit))}]))
display(removed_flow_audit.head(20))


## Summary

以下は観測値の要約です。除外理由や通信/applicationの意味を自動的に断定するものではありません。

In [ ]:
removed_tcp_patterns = tcp_pattern_comparison.loc[tcp_pattern_comparison['group'].eq('Broad removed')].sort_values(['flow count', 'observed_tcp_pattern'], ascending=[False, True])
removed_ports = receiver_port_comparison.loc[receiver_port_comparison['group'].eq('Broad removed')].sort_values(['flow count', 'receiver port'], ascending=[False, True])
removed_roles = syn_role_comparison.loc[syn_role_comparison['group'].eq('Broad removed')].set_index('Prefix-side SYN role')['ratio within observed-SYN TCP [%]']
summary = pd.DataFrame([{
    'prefix': TARGET_PREFIX,
    'Raw flow count': len(raw_target_flows),
    'Broad removed count': len(broad_removed_flows),
    'Broad removed ratio [%]': ratio_percent(raw_target_flows['broad_removed']),
    'Raw median packet_count': raw_median_packet_count,
    'Broad median packet_count': remaining_median_packet_count,
    'median change rate [%]': ((remaining_median_packet_count - raw_median_packet_count) / raw_median_packet_count * 100) if pd.notna(raw_median_packet_count) and raw_median_packet_count != 0 else np.nan,
    'removed median packet_count': numeric_median(broad_removed_flows, 'packet_count'),
    'remaining median packet_count': remaining_median_packet_count,
    'removed one-packet ratio [%]': ratio_percent(broad_removed_flows['packet_count'].eq(1)),
    'dominant removed TCP pattern': removed_tcp_patterns.iloc[0]['observed_tcp_pattern'] if len(removed_tcp_patterns) else np.nan,
    'dominant removed receiver port': removed_ports.iloc[0]['receiver port'] if len(removed_ports) else np.nan,
    'unique removed receiver IP count': removed_syn_tcp['initial_syn_receiver_ip'].nunique(),
    'Prefix-side SYN sender ratio [%]': removed_roles.get('initial SYN sender', np.nan),
    'Prefix-side SYN receiver ratio [%]': removed_roles.get('initial SYN receiver', np.nan),
}])
display(summary)
